In [1]:
import pkg_resources
pkg_resources.require("geopandas==0.13.0")
#pkg_resources.require("pandas==1.4.4")
pkg_resources.require("folium==0.14.0")
#pkg_resources.require("ipywidgets==8.0.4")

[folium 0.14.0 (c:\users\cday\anaconda3\lib\site-packages),
 requests 2.32.1 (c:\users\cday\anaconda3\lib\site-packages),
 numpy 1.25.1 (c:\users\cday\anaconda3\lib\site-packages),
 jinja2 3.1.4 (c:\users\cday\anaconda3\lib\site-packages),
 branca 0.6.0 (c:\users\cday\anaconda3\lib\site-packages),
 certifi 2023.5.7 (c:\users\cday\anaconda3\lib\site-packages),
 urllib3 1.26.18 (c:\users\cday\anaconda3\lib\site-packages),
 idna 3.7 (c:\users\cday\anaconda3\lib\site-packages),
 charset-normalizer 3.3.2 (c:\users\cday\anaconda3\lib\site-packages),
 MarkupSafe 2.1.3 (c:\users\cday\anaconda3\lib\site-packages),
 jinja2 3.1.4 (c:\users\cday\anaconda3\lib\site-packages)]

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium as fm
import ipywidgets as widgets
from ipywidgets import interact

c:\Users\cday\Anaconda3\lib\site-packages\geopandas\_compat.py:124: UserWarning: The Shapely GEOS version (3.8.0-CAPI-1.13.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
C:\Users\cday\AppData\Local\Temp\ipykernel_32328\3136618975.py:1: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should be smooth. However, if you are 

In [3]:
# Define filters
strLinksFilter = "" #"MAG_LINK == 0"
strNodesFilter = "" #"MAG_NODE == 0"

bFillNa= True

# Define precision settings for specific fields, e.g., DISTANCE
precision_settings = {'DISTANCE': 0.0001, 'X': 0.1, 'Y': 0.1}

skipFields = ['geometry', 'TAZID']

In [4]:
# read net 1
gdfNt1Links = gpd.read_file('input/WFv911_MasterNet - Link.shp')
print(gdfNt1Links.shape)
print(gdfNt1Links.crs)

gdfNt1Nodes = gpd.read_file('input/WFv911_MasterNet - Node.shp')
print(gdfNt1Nodes.shape)
print(gdfNt1Nodes.crs)

(57810, 127)
EPSG:26912
(21806, 33)
EPSG:26912


In [5]:
# read net 2
gdfNt2Links = gpd.read_file('input/WFv920_MasterNet - Link.shp')
print(gdfNt2Links.shape)
print(gdfNt2Links.crs)

gdfNt2Nodes = gpd.read_file('input/WFv920_MasterNet - Node.shp')
print(gdfNt2Nodes.shape)
print(gdfNt2Nodes.crs)

(57810, 127)
EPSG:26912
(21806, 33)
EPSG:26912


In [6]:
# Apply filters
gdfNt1LinksFiltered = gdfNt1Links#.query(strLinksFilter)
print(gdfNt1LinksFiltered.shape)

gdfNt1NodesFiltered = gdfNt1Nodes#.query(strNodesFilter)
print(gdfNt1NodesFiltered.shape)

# Apply filters
gdfNt2LinksFiltered = gdfNt2Links#.query(strLinksFilter)
print(gdfNt2LinksFiltered.shape)

gdfNt2NodesFiltered = gdfNt2Nodes#.query(strNodesFilter)
print(gdfNt2NodesFiltered.shape)

(57810, 127)
(21806, 33)
(57810, 127)
(21806, 33)


In [7]:
# Function to fill NaN values in a GeoDataFrame
def fill_nan_values(gdf):
    # Create a copy of the GeoDataFrame to avoid modifying the original DataFrame
    gdf_copy = gdf.copy()
    
    # Fill NaN in text fields with ''
    text_columns = gdf_copy.select_dtypes(include=['object']).columns
    for column in text_columns:
        gdf_copy.loc[:, column] = gdf_copy.loc[:, column].fillna('')
    
    # Fill NaN in numeric fields with 0
    numeric_columns = gdf_copy.select_dtypes(include=[np.number]).columns
    for column in numeric_columns:
        gdf_copy.loc[:, column] = gdf_copy.loc[:, column].fillna(0)

    return gdf_copy

# fill NAs for better comparison of empty values
if bFillNa:
    gdfNt1LinksFiltered = fill_nan_values(gdfNt1LinksFiltered)
    gdfNt1NodesFiltered = fill_nan_values(gdfNt1NodesFiltered)
    gdfNt2LinksFiltered = fill_nan_values(gdfNt2LinksFiltered)
    gdfNt2NodesFiltered = fill_nan_values(gdfNt2NodesFiltered)


In [8]:
def compare_gdfs(gdf1, gdf2, keys, precision_settings=None, skipFields=None):
    if not isinstance(keys, list):
        keys = [keys]
    if precision_settings is None:
        precision_settings = {}
    if skipFields is None:
        skipFields = []

    common = gdf1.merge(gdf2, on=keys, suffixes=('_1', '_2'))

    # Initialize a list to collect individual row differences
    differences_data = []

    for column in gdf1.columns:
        if column in skipFields or column in keys:  # Skip specified fields and key columns
            continue
        if f'{column}_1' in common.columns and f'{column}_2' in common.columns:  # Ensure the column exists in both dataframes
            if column in precision_settings:
                tolerance = precision_settings[column]
                diff = common[abs(common[f'{column}_1'] - common[f'{column}_2']) > tolerance]
            else:
                diff = common[common[f'{column}_1'] != common[f'{column}_2']]

            # Add row differences to the list
            for _, row in diff.iterrows():
                difference_dict = {key: row[key] for key in keys}  # Create a separate entry for each key
                difference_dict.update({
                    'field': column,
                    'value_in_1': row[f'{column}_1'],
                    'value_in_2': row[f'{column}_2']
                })
                differences_data.append(difference_dict)
    print(differences_data)
    # Convert the list of differences to a DataFrame
    differences_df = pd.DataFrame(differences_data)

    # Identify records only present in one of the GeoDataFrames
    only_in_gdf1 = gdf1[~gdf1[keys].apply(tuple, 1).isin(gdf2[keys].apply(tuple, 1))].drop_duplicates()
    only_in_gdf2 = gdf2[~gdf2[keys].apply(tuple, 1).isin(gdf1[keys].apply(tuple, 1))].drop_duplicates()

    return differences_df, only_in_gdf1, only_in_gdf2


In [9]:
# Compare Links
diff_links, only_in_1_links, only_in_2_links = compare_gdfs(gdfNt1LinksFiltered, gdfNt2LinksFiltered, ['A', 'B'], precision_settings)
# Compare Nodes
diff_nodes, only_in_1_nodes, only_in_2_nodes = compare_gdfs(gdfNt1NodesFiltered, gdfNt2NodesFiltered, ['N'], precision_settings)

[{'A': 26076, 'B': 26106, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26106, 'B': 26131, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26108, 'B': 26075, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26130, 'B': 26108, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26131, 'B': 26154, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26148, 'B': 26130, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26154, 'B': 26191, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26186, 'B': 26148, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26191, 'B': 26197, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26193, 'B': 26186, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26197, 'B': 26243, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26240, 'B': 26193, 'field': 'LN23_32', 'value_in_1': 2, 'value_in_2': 3}, {'A': 26243, 'B': 26260, 'field': 'LN23

In [10]:
diff_categories = ['Different', 'In Net1 and Not Net2', 'In Net2 and Not Net1']

# functions to create map and color scale
def get_map(lat=40.7238, long=-111.4585, zoom_start=9):
    f = fm.Figure(width =800, height=1600)
    return fm.Map(location=[lat,long], zoom_start=zoom_start,
                     tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}', #'https://{s}.basemaps.cartocdn.com/rastertiles/voyager_nolabels/{z}/{x}/{y}{r}.png', # for other basemaps: https://leaflet-extras.github.io/leaflet-providers/preview/
                     attr='Esri Ocean Basemap').add_to(f)
from folium.plugins import MarkerCluster
def update_map(diffCat, nodeChecked, linkChecked, alllinkChecked, linkField, nodeField):
    map = get_map()
    
    if diffCat == 'Different':
        if linkChecked:
            _gdfLinks = pd.merge(gdfNt1LinksFiltered, diff_links[diff_links['field'].isin(linkField)], on=('A','B'))
            _fieldsLinks = ['A','B','value_in_1','value_in_2']
        if nodeChecked:
            _gdfNodes = pd.merge(gdfNt1NodesFiltered, diff_nodes[diff_nodes['field']==nodeField], on=('N'))
            _fieldsNodes = ['N','value_in_1','value_in_2']
    elif diffCat == 'In Net1 and Not Net2': 
        if linkChecked:
            _gdfLinks = gpd.GeoDataFrame(only_in_1_links, geometry='geometry', crs="EPSG:32612")
            _fieldsLinks = ['A','B']
        if nodeChecked:
            _gdfNodes = gpd.GeoDataFrame(only_in_1_nodes, geometry='geometry', crs="EPSG:32612")
            _fieldsNodes = ['N']
    elif diffCat=='In Net2 and Not Net1':
        if linkChecked:
            _gdfLinks = gpd.GeoDataFrame(only_in_2_links, geometry='geometry', crs="EPSG:32612")
            _fieldsLinks = ['A','B']
        if nodeChecked:
            _gdfNodes = gpd.GeoDataFrame(only_in_2_nodes, geometry='geometry', crs="EPSG:32612")
            _fieldsNodes = ['N']

    if alllinkChecked and not gdfNt1LinksFiltered.empty:
        v900_gdfLinks = gpd.GeoDataFrame(gdfNt1LinksFiltered, geometry='geometry', crs="EPSG:32612")
        v900_fieldsLinks = ['A','B']

        fm.GeoJson(
            v900_gdfLinks,
            style_function=lambda x: {'color': '#5FA052', 'weight': 0.5},  # 7F7F7F
            tooltip=fm.GeoJsonTooltip(fields=v900_fieldsLinks, localize=True)
        ).add_to(map)

    if linkChecked and not _gdfLinks.empty:
        fm.GeoJson(
            _gdfLinks,
            style_function=lambda x: {'color': '#1E90FF', 'weight': 8},
            tooltip=fm.GeoJsonTooltip(
                fields=_fieldsLinks,
                localize=True
            )
        ).add_to(map)

    if nodeChecked and not _gdfNodes.empty:
        fm.GeoJson(
            _gdfNodes,
                 marker = fm.CircleMarker(
                    radius = 4, # Radius in metres
                    weight = 0, #outline weight
                    fill_color = '#1E90FF', 
                    fill_opacity = 1
                ),
            tooltip=fm.GeoJsonTooltip(
                fields=_fieldsNodes,
                localize=True
            )
        ).add_to(map)

    map.caption = diffCat + ' - ' + str(linkField)  + nodeField

    return map

In [11]:
diff_nodes = diff_links

In [12]:
diffCat_dropdown   = widgets.Dropdown(options=diff_categories, value=diff_categories[0], description='Diff Category:')
linkField_dropdown = widgets.SelectMultiple(options=diff_links['field'].drop_duplicates() , value=[diff_links['field'].drop_duplicates()[0]] , description='Link Fields:'  )
nodeField_dropdown = widgets.Dropdown(options=diff_nodes['field'].drop_duplicates() , value=diff_nodes['field'].drop_duplicates()[0] , description='Node Fields:'  )

link_check = widgets.Checkbox(value=True, description='Show Links')
node_check = widgets.Checkbox(value=False, description='Show Nodes')
alllink_check  = widgets.Checkbox(value=False, description='Show Links Base Layer')

#Display the widgets and plot
##display(widgets.HBox([mdlcom_des_dropdown, mode_desc_dropdown]))
interact(update_map, diffCat   = diffCat_dropdown,
                     nodeChecked = node_check,
                     linkChecked = link_check,
                     alllinkChecked = alllink_check,
                     nodeField = nodeField_dropdown,
                     linkField = linkField_dropdown,)



interactive(children=(Dropdown(description='Diff Category:', options=('Different', 'In Net1 and Not Net2', 'In…

<function __main__.update_map(diffCat, nodeChecked, linkChecked, alllinkChecked, linkField, nodeField)>

In [12]:
# TO DO
#   make map look better for screen shots (different colors/basemap?)
#   run Chad's Cube script and screen shot those maps

# Add column for differences to network 2

In [12]:
display (diff_nodes, only_in_1_nodes, only_in_2_nodes)

,N,field,value_in_1,value_in_2
0,920,X,404989.69383,404908.0
1,15026,X,423285.6071,423274.96654
2,20711,X,422012.31494,422010.22747
3,20831,X,413265.1785,413278.56928
4,23181,X,417445.85715,417445.50846
...,...,...,...,...
109,29832,geometry,POINT (425358.9667 4493760.4045),POINT (425358.9667352129 4493760.40449765)
110,29833,geometry,POINT (425478.31 4491917.7577),POINT (425478.3099784562 4491917.757679406)
111,29834,geometry,POINT (425567.4926 4492045.7778),POINT (425567.4925535796 4492045.777827566)
112,29835,geometry,POINT (425576.1231 4492229.8967),POINT (425576.1231253656 4492229.896692337)


,N,X,Y,GEOGKEY,EXTERNAL,HOTZN,TAZID,NODENAME,PNR_2015,PNR_2019,...,FARZN23_42,FARZN23_50,FARE23_32U,FARE23_42U,FARE23_50U,MAG_NODE,WFRC_NODE,FLAG,FLAG_TXT,geometry


,N,X,Y,GEOGKEY,EXTERNAL,HOTZN,TAZID,NODENAME,PNR_2015,PNR_2019,...,FARZN_2028,FARZN23_32,FARZN23_42,FARZN23_50,FARE23_32U,FARE23_42U,FARE23_50U,MAG_NODE,WFRC_NODE,geometry


In [13]:
diff_nodes[diff_nodes['field']=='X']

dfDiffList = diff_nodes[diff_nodes['field']=='X'].drop(columns=['field','value_in_1','value_in_2'])
dfDiffList['CompareFlag'] = 'Diff'
dfDiffList

,N,CompareFlag
0,920,Diff
1,15026,Diff
2,20711,Diff
3,20831,Diff
4,23181,Diff
5,26210,Diff
6,69807,Diff


In [14]:
dfNewList = only_in_2_nodes[['N']].copy()
dfNewList['CompareFlag'] = 'NewInV902'
dfNewList

,N,CompareFlag


In [15]:
dfList = pd.concat([dfNewList, dfDiffList]).reset_index().drop(columns=('index'))
dfList

,N,CompareFlag
0,920,Diff
1,15026,Diff
2,20711,Diff
3,20831,Diff
4,23181,Diff
5,26210,Diff
6,69807,Diff


In [16]:
gdfNt2NodesFiltered_withCompareFlag = pd.merge(gdfNt2NodesFiltered, dfList, on='N', how='left').fillna('')
gdfNt2NodesFiltered_withCompareFlag

,N,X,Y,GEOGKEY,EXTERNAL,HOTZN,TAZID,NODENAME,PNR_2015,PNR_2019,...,FARZN23_32,FARZN23_42,FARZN23_50,FARE23_32U,FARE23_42U,FARE23_50U,MAG_NODE,WFRC_NODE,geometry,CompareFlag
0,1,411935.0000,4605642.000,411935_4605642,0,0,1,,0,0,...,0,0,0,0,0,0,0,1,POINT (411935.000 4605642.000),
1,2,412641.0000,4606013.000,412641_4606013,0,0,2,,0,0,...,0,0,0,0,0,0,0,1,POINT (412641.000 4606013.000),
2,3,408993.0000,4603544.000,408993_4603544,0,0,3,,0,0,...,0,0,0,0,0,0,0,1,POINT (408993.000 4603544.000),
3,4,410051.0000,4604806.000,410051_4604806,0,0,4,,0,0,...,0,0,0,0,0,0,0,1,POINT (410051.000 4604806.000),
4,5,413394.0000,4603600.000,413394_4603600,0,0,5,,0,0,...,0,0,0,0,0,0,0,1,POINT (413394.000 4603600.000),
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21435,95061,431050.6881,4422110.715,431050_4422110,0,0,3469,,0,0,...,0,0,0,0,0,0,1,0,POINT (431050.688 4422110.715),
21436,95062,430713.6121,4421517.072,430713_4421517,0,0,3469,,0,0,...,0,0,0,0,0,0,1,0,POINT (430713.612 4421517.072),
21437,95063,452850.1772,4468529.559,452850_4468529,0,0,3508,,0,0,...,0,0,0,0,0,0,1,0,POINT (452850.177 4468529.559),
21438,95064,473181.5500,4444171.200,473181_4444171,0,0,3536,,0,0,...,0,0,0,0,0,0,1,0,POINT (473181.550 4444171.200),


In [17]:
gdfNt2NodesFiltered_withCompareFlag.to_file('WFv910_MasterNet_20240430_Nodes_withCompareFlag.shp')


C:\Users\cday\AppData\Local\Temp\ipykernel_17148\4122579967.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdfNt2NodesFiltered_withCompareFlag.to_file('WFv902_MasterNet_20240430_Nodes_withCompareFlag.shp')


## Summary Statistics

In [21]:
gdfTaz = gpd.read_file(r"D:\GitHub\WF-TDM-v9x\1_Inputs\1_TAZ\WFv900_TAZ.shp")
gdfTaz = gdfTaz[['geometry','CITY_NAME','CO_NAME']]

### Different SEGIDs

In [22]:
linkField = 'SEGID'
gdfLinksDifferent = pd.merge(gdfNt1LinksFiltered, diff_links[diff_links['field']==linkField], on=('A','B'))

gdfLinksDifferentC = gpd.GeoDataFrame(gdfLinksDifferent.drop(columns=['geometry']), geometry=gdfLinksDifferent.geometry.centroid)
gdfLinksDifferentSegs = gpd.sjoin(gdfLinksDifferentC, gdfTaz, how='left', op='within')
gdfLinksDifferentSegs = gdfLinksDifferentSegs.drop_duplicates(keep='first')

dfLinksDifferentSegs = pd.DataFrame(gdfLinksDifferentSegs)
dfLinksDifferentSegs = dfLinksDifferentSegs[['LINKID','SEGID','CITY_NAME','CO_NAME','field','value_in_1','value_in_2']]
display(dfLinksDifferentSegs)

c:\Users\cday\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


,LINKID,SEGID,CITY_NAME,CO_NAME,field,value_in_1,value_in_2
0,62354_71559,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,
1,62361_68943,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,
2,62361_71560,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,
3,66850_66851,3044_000.0,Provo,UTAH,SEGID,3044_000.0,
4,66851_66850,0189_001.1,Provo,UTAH,SEGID,0189_001.1,
5,68941_68944,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,
6,68943_62361,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,
7,68943_68944,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,
8,68944_68941,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,
9,68944_68943,0145_005.8,American Fork,UTAH,SEGID,0145_005.8,


In [23]:
city_counts_diff_segs = dfLinksDifferentSegs['CITY_NAME'].value_counts()
display(city_counts_diff_segs)

CITY_NAME
American Fork    10
Provo             2
Name: count, dtype: int64

In [24]:
county_counts_diff_segs = dfLinksDifferentSegs['CO_NAME'].value_counts()
display(county_counts_diff_segs)

CO_NAME
UTAH    12
Name: count, dtype: int64

### New SEGIDS

In [25]:
dfNewSegs = dfLinksDifferentSegs[(dfLinksDifferentSegs['value_in_1'] == '') & (dfLinksDifferentSegs['value_in_2'] != '')]
dfNewSegs

,LINKID,SEGID,CITY_NAME,CO_NAME,field,value_in_1,value_in_2


In [26]:
city_counts_new_segs = dfNewSegs['CITY_NAME'].value_counts()
display(city_counts_new_segs)

Series([], Name: count, dtype: int64)

### Renamed SEGIDs

In [27]:
dfRenameSegs = dfLinksDifferentSegs[(dfLinksDifferentSegs['value_in_1'] != '') & (dfLinksDifferentSegs['value_in_2'] != '')]
dfRenameSegs

,LINKID,SEGID,CITY_NAME,CO_NAME,field,value_in_1,value_in_2


In [28]:
city_counts_rename_segs = dfRenameSegs['CITY_NAME'].value_counts()
display(city_counts_rename_segs)

Series([], Name: count, dtype: int64)

### Renamed LINKIDs

In [29]:
linkField = 'LINKID'
gdfLinksDifferent = pd.merge(gdfNt1LinksFiltered, diff_links[diff_links['field']==linkField], on=('A','B'))

gdfLinksDifferentC = gpd.GeoDataFrame(gdfLinksDifferent.drop(columns=['geometry']), geometry=gdfLinksDifferent.geometry.centroid)
gdfLinksDifferentCity = gpd.sjoin(gdfLinksDifferentC, gdfTaz, how='left', op='within')
gdfLinksDifferentCity = gdfLinksDifferentCity.drop_duplicates(keep='first')

dfLinksDifferentCity = pd.DataFrame(gdfLinksDifferentCity)
dfLinksDifferentCity = dfLinksDifferentCity.rename(columns={'value_in_1':'value_in_v900', 'value_in_2':'value_in_v901'})
dfLinksDifferentCity = dfLinksDifferentCity[['LINKID','CITY_NAME','CO_NAME','value_in_v900','value_in_v901']]
display(dfLinksDifferentCity.sort_values('CITY_NAME'))

c:\Users\cday\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


,LINKID,CITY_NAME,CO_NAME,value_in_v900,value_in_v901
29,69834_69833,Draper,UTAH,69834_69833,71595_71594
27,69833_69834,Draper,UTAH,69833_69834,71594_71595
26,69833_69834,Draper,UTAH,69833_69834,69834_71595
28,69834_69833,Draper,UTAH,69834_69833,71595_69834
9,29315_29331,Pleasant View,WEBER,29315_29331,29331_29344
10,29331_29315,Pleasant View,WEBER,29331_29315,29344_29331
25,23060_22953,Sandy,SALT LAKE,23060_22953,23158_23060
24,22953_23060,Sandy,SALT LAKE,22953_23060,23060_23158
23,22953_23060,Sandy,SALT LAKE,22953_23060,23060_23158
22,23060_22953,Sandy,SALT LAKE,23060_22953,23158_23060


In [30]:
county_counts_rename_links = dfLinksDifferentCity['CO_NAME'].value_counts()
display(county_counts_rename_links)

CO_NAME
SALT LAKE    24
UTAH          4
WEBER         2
Name: count, dtype: int64

### New LINKIDs

In [31]:
only_in_2_links
only_in_2_linksC = gpd.GeoDataFrame(only_in_2_links.drop(columns=['geometry']), geometry=only_in_2_links.geometry.centroid)
only_in_2_linksCity = gpd.sjoin(only_in_2_linksC, gdfTaz, how='left', op='within')
only_in_2_linksCity = only_in_2_linksCity.drop_duplicates(keep='first')
only_in_2_linksCity[['LINKID','CITY_NAME']]

c:\Users\cday\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


,LINKID,CITY_NAME


In [32]:
city_counts_new_links = only_in_2_linksCity['CITY_NAME'].value_counts()
display(city_counts_new_links)

Series([], Name: count, dtype: int64)

### Repositioned Nodes

In [33]:
nodeField = 'X'
gdfNodesDifferent = pd.merge(gdfNt1NodesFiltered, diff_nodes[diff_nodes['field']==nodeField], on=('N'))

gdfNodesDifferentCity = gpd.sjoin(gdfNodesDifferent, gdfTaz, how='left', op='within')
gdfNodesDifferentCity = gdfNodesDifferentCity.drop_duplicates(keep='first')

dfNodesDifferentCity = pd.DataFrame(gdfNodesDifferentCity)
dfNodesDifferentCity = dfNodesDifferentCity.rename(columns={'value_in_1':'value_in_v900', 'value_in_2':'value_in_v901'})
dfNodesDifferentCity = dfNodesDifferentCity[['N','CITY_NAME','CO_NAME','value_in_v900','value_in_v901']]
display(dfNodesDifferentCity.sort_values('CITY_NAME'))



c:\Users\cday\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


,N,CITY_NAME,CO_NAME,value_in_v900,value_in_v901
2,20711,Bluffdale,SALT LAKE,422012.31494,422010.22747
1,15026,Draper,SALT LAKE,423285.6071,423274.96654
6,69807,Lehi,UTAH,422590.5834,422601.85908
5,26210,North Salt Lake,DAVIS,422142.0,422257.18165
0,920,SL County West,SALT LAKE,404989.69383,404908.0
3,20831,South Jordan,SALT LAKE,413265.1785,413278.56928
4,23181,West Valley City,SALT LAKE,417445.85715,417445.50846


In [34]:
city_counts_rename_nodes = dfNodesDifferentCity['CITY_NAME'].value_counts()
display(city_counts_rename_nodes)

CITY_NAME
SL County West      1
Draper              1
Bluffdale           1
South Jordan        1
West Valley City    1
North Salt Lake     1
Lehi                1
Name: count, dtype: int64

### New Nodes

In [35]:
only_in_2_nodes
only_in_2_nodesCity = gpd.sjoin(only_in_2_nodes, gdfTaz, how='left', op='within')
only_in_2_nodesCity[['N','CITY_NAME']]

c:\Users\cday\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


,N,CITY_NAME


In [36]:
city_counts_new_nodes = only_in_2_nodesCity['CITY_NAME'].value_counts()
display(city_counts_new_nodes)

Series([], Name: count, dtype: int64)

## Link Attributes

In [37]:
fields_to_process = ['DISTANCE', 'STREET','ONEWAY','DIRECTION',
                     #'LN_2015',
                     'LN_2019', "LN_2021",'LN_2023', 'LN_2028', 'LN23_32', 'LN23_42', 'LN23_50', 'LN23_32UF', 'LN23_42UF', 'LN23_50UF', 'LN23_50UFM',
                     'FT_2015', 'FT_2019', "FT_2021",'FT_2023', 'FT_2028', 'FT23_32', 'FT23_42', 'FT23_50', 'FT23_32UF', 'FT23_42UF', 'FT23_50UF']

concatenated_links = pd.DataFrame()

for field in fields_to_process:
    diff_links_field = diff_links[diff_links['field'] == field]
    merged_links = pd.merge(gdfNt1LinksFiltered, diff_links_field, on=('A', 'B')).drop(columns={'geometry'})
    merged_links = merged_links[['A', 'B', 'LINKID', 'field', 'value_in_1', 'value_in_2']]
    merged_links['source_field'] = field
    concatenated_links = pd.concat([concatenated_links, merged_links], ignore_index=True)

print(concatenated_links.head())

def nest_values(group):
    return pd.Series({
        'A': group['A'].iloc[0],
        'B': group['B'].iloc[0],
        'LINKID': group['LINKID'].iloc[0],
        'field_list': group['field'].tolist(),
        'value_in_1': group['value_in_1'].iloc[0],
        'value_in_2': group['value_in_2'].iloc[0]
    })

# Group by specified columns and apply the custom aggregation function
nested_df = concatenated_links.groupby(['A', 'B', 'LINKID', 'value_in_1', 'value_in_2']).apply(nest_values).reset_index(drop=True)

# Rename the aggregated column to 'field'
nested_df = nested_df.rename(columns={'field_list': 'field'})

       A      B       LINKID     field value_in_1 value_in_2 source_field
0  10006  10060  10006_10007  DISTANCE        0.0      0.459     DISTANCE
1  10007  10060  10007_10006  DISTANCE        0.0     1.3898     DISTANCE
2  10060  10006  10007_10006  DISTANCE        0.0      0.459     DISTANCE
3  10060  10007  10006_10007  DISTANCE        0.0     1.3898     DISTANCE
4  15026  15027  15026_15027  DISTANCE    0.03265    0.08237     DISTANCE


C:\Users\cday\AppData\Local\Temp\ipykernel_17148\2269169655.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nested_df = concatenated_links.groupby(['A', 'B', 'LINKID', 'value_in_1', 'value_in_2']).apply(nest_values).reset_index(drop=True)


In [38]:
field_count_summary_nested = nested_df['field'].value_counts().reset_index()
field_count_summary_nested.columns = ['field', 'count']
field_count_summary_nested

,field,count
0,"[LN23_32, LN23_42, LN23_50, LN23_32UF, LN23_42...",213
1,"[FT23_32, FT23_42, FT23_50, FT23_32UF, FT23_42...",40
2,"[LN23_42, LN23_32UF]",37
3,"[LN_2028, LN23_32]",34
4,"[LN23_50UF, LN23_50UFM]",33
5,[STREET],32
6,"[LN23_32, LN23_42, LN23_50, LN23_32UF, LN23_42UF]",31
7,"[LN_2028, LN23_32, LN23_42, LN23_32UF]",25
8,[DISTANCE],20
9,[LN23_50],14


In [39]:
field_count_summary = concatenated_links['field'].value_counts().reset_index()
field_count_summary.columns = ['field', 'count']
field_count_summary

,field,count
0,LN23_42,327
1,LN23_32UF,326
2,LN23_32,321
3,LN23_50,276
4,LN23_42UF,264
5,LN23_50UF,262
6,LN23_50UFM,262
7,LN_2028,86
8,FT23_42UF,52
9,FT23_50,52
